# 🏆 VAR 2026 — Digital Twin trạm BTS: 3DGS Pipeline (Kaggle GPU Notebook)

> **Bài toán:** Novel View Synthesis — Digital Twin cho hạ tầng viễn thông BTS  
> **Mô hình:** 3D Gaussian Splatting (3DGS v2.5.0 Self-Contained) với Anti-Aliasing  
> **Công thức Metric:** $Score = 0.4 \times (1 - \text{LPIPS}) + 0.3 \times \text{SSIM} + 0.3 \times \text{PSNR}_{norm}$  

Notebook này được tối ưu 100% để chạy trên **Kaggle GPU (Tesla T4 16GB)** hoặc **Google Colab / Linux GPU Server**. Tự động phát hiện dữ liệu input, biên dịch C++ extensions, render ảnh RGB chân thực mượt mà (với 2D Gaussian Ellipse Alpha-Blending) và tạo file nộp bài `submission_round1.zip`.

---
## 🛠️ Bước 1: Khởi Tạo Môi Trường & Cấu Hình `sys.path`

Tự động thêm đường dẫn làm việc và các thư mục `src/`, `src/_3dgs/` vào `sys.path`.

In [ ]:
import sys, os
from pathlib import Path

# Phát hiện vị trí chạy (Kaggle / Local / Colab)
NOTEBOOK_DIR = Path(".").resolve()
if (Path("/kaggle/working")).exists():
    PROJECT_ROOT = Path("/kaggle/working")
else:
    PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent

SRC_DIR = PROJECT_ROOT / "src"
GS_DIR = SRC_DIR / "_3dgs"

for p in [PROJECT_ROOT, SRC_DIR, GS_DIR]:
    if str(p) not in sys.path and p.exists():
        sys.path.insert(0, str(p))

print("✅ Cấu hình sys.path thành công:")
for p in sys.path[:4]:
    print(f"  - {p}")


In [ ]:
# Kiểm tra GPU Accelerator trên Kaggle / Local
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


---
## 📦 Bước 2: Cài Đặt Tools & Build C++ CUDA Extensions

Cài đặt COLMAP, Ninja build và biên dịch 3 CUDA submodules (`simple-knn`, `diff-gaussian-rasterization`, `fused-ssim`).

In [ ]:
# 1. Cài đặt hệ thống (Nghiêm cấm bỏ qua trên Kaggle)
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq colmap ninja-build > /dev/null 2>&1
!pip install --upgrade "pip<27" "setuptools<82" wheel ninja -q
!pip install plyfile tqdm opencv-python pillow scipy rich -q


In [ ]:
# 2. Build & Install C++ CUDA extensions cho 3DGS (Chạy mất ~30 giây trên GPU T4)
!pip install -e src/_3dgs/submodules/simple-knn --no-build-isolation
!pip install -e src/_3dgs/submodules/diff-gaussian-rasterization --no-build-isolation
!pip install -e src/_3dgs/submodules/fused-ssim --no-build-isolation


In [ ]:
# 3. Xác nhận các C++ extensions đã load mượt mà
try:
    from simple_knn._C import distCUDA2
    from diff_gaussian_rasterization import _C as diff_c
    from fused_ssim import fused_ssim
    print("🎉 Tất cả C++ CUDA extensions đã biên dịch & sẵn sàng!")
except ImportError as e:
    print(f"❌ Lỗi import C++ submodules: {e}")


---
## 🔍 Bước 3: Tự Động Tìm Dataset & Validating Data

Tự động phát hiện các tập Kaggle datasets đầu vào (bao gồm dataset scenes & pre-trained checkpoints).

In [ ]:
# Import trực tiếp các module nguồn trong dự án
from config import SCENES, VARIANTS, DATA_DIR, OUTPUT_DIR, get_scene_variant
from main import validate

print(f"✅ Danh sách scenes: {SCENES}")
print(f"✅ Biến thể huấn luyện: {list(VARIANTS.keys())}")

# Kiểm tra toàn vẹn dữ liệu
validate(SCENES)


In [ ]:
# Dry-run kiểm tra toàn bộ pipeline
!python src/main.py --dry-run


---
## 🎨 Bước 4: Render Ảnh RGB Chân Thực Từ Pre-trained Checkpoint (`HCM0421`)

Render 60 test poses cho scene `HCM0421` từ checkpoint `hcm0421-checkpoint-25000` với đầy đủ thuật toán nở hạt elip 2D (Alpha-blended Gaussian Splatting) bằng C++ extension.

In [ ]:
# Render ảnh RGB chân thực từ Pre-trained Checkpoint
import csv, math, json, os
import numpy as np, torch, torchvision
from pathlib import Path

# Kiểm tra vị trí checkpoint trên Kaggle hoặc local
ckpt_candidates = [
    Path("/kaggle/input/hcm0421-checkpoint-25000/chkpnt25000.pth"),
    Path("/kaggle/input/datasets/dangthtai/hcm0421-checkpoint-25000/chkpnt25000.pth"),
    PROJECT_ROOT / "output" / "checkpoints" / "HCM0421_25k" / "chkpnt25000.pth"
]
ckpt_file = next((p for p in ckpt_candidates if p.exists()), None)

if ckpt_file:
    print(f"🎯 Tìm thấy checkpoint: {ckpt_file}")
    # Tạo model dir để render.py nhận diện
    model_dir = OUTPUT_DIR / "models" / "HCM0421" / "pretrained_25k"
    pc_dir = model_dir / "point_cloud" / "iteration_25000"
    pc_dir.mkdir(parents=True, exist_ok=True)
    
    # Render test poses với C++ Extension
    !python src/render.py --scene HCM0421 --variant fast
else:
    print("ℹ️ Chưa tìm thấy pre-trained checkpoint. Tiến hành bước train ở Bước 5!")


---
## ⚡ Bước 5: Fast Run & Benchmark (1 Scene - 7k iters ~3 phút)

Chạy thử nghiệm nhanh biến thể `fast` trên 1 scene để xác nhận toàn bộ quy trình train $\rightarrow$ render $\rightarrow$ eval $\rightarrow$ package.

In [ ]:
# Train fast variant trên 1 scene
!python src/main.py --scenes bonsai --variant fast


---
## 🚀 Bước 6: Full Top-1 Pipeline Run (Tất Cả Scene BTS Viettel)

Huấn luyện đầy đủ các biến thể (variants), áp dụng Gaussian Compact Merge, Test-Time Adaptation (TTA), Perceptual Fine-Tuning (LPIPS/DINOv2) và Ensemble 5-tín hiệu.

In [ ]:
# Chạy Full Top-1 Pipeline cho các scene trạm BTS
!python src/main.py --scenes HCM0421 HCM0539 HCM0540 HCM0644 HCM0674 --all-variants --compact --tta --perceptual


---
## 📦 Bước 7: Tạo File Nộp Bài `submission_round1.zip` & Dowload

Tự động đóng gói tất cả ảnh kết quả thành file ZIP chuẩn nộp bài.

In [ ]:
# Chạy phase đóng gói package.py
!python src/package.py --source final


In [ ]:
# Kiểm tra chi tiết file submission_round1.zip
import zipfile
from pathlib import Path

sub_zip = PROJECT_ROOT / "src" / "submissions" / "submission_round1.zip"
if not sub_zip.exists():
    sub_zip = Path("/kaggle/working/submission_round1.zip")

if sub_zip.exists():
    size_mb = sub_zip.stat().st_size / (1024 * 1024)
    print(f"🎉 FILE NỘP BÀI ĐÃ SẴN SÀNG: {sub_zip} ({size_mb:.2f} MB)")
    with zipfile.ZipFile(sub_zip, "r") as z:
        files = z.namelist()
        print(f"Tổng số ảnh trong tệp ZIP: {len(files)}")
        print("5 file ảnh mẫu đầu tiên:", files[:5])
else:
    print("❌ Chưa tìm thấy file submission_round1.zip. Hãy chạy lại Bước 7!")
